In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Carregar o dataset de câncer de mama (um dataset real de classificação binária)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Dataset carregado com sucesso!")

Dataset carregado com sucesso!


### 1. Exploração Inicial dos Dados

In [2]:
print("Primeiras 5 linhas do DataFrame X:")
display(X.head())

print("Informações sobre as colunas (tipos de dados, não-nulos):")
X.info()

print("Estatísticas descritivas básicas:")
display(X.describe())

Primeiras 5 linhas do DataFrame X:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Informações sobre as colunas (tipos de dados, não-nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error            

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


### 2. Pré-processamento de Dados

Vamos dividir os dados em conjuntos de treino e teste e, em seguida, aplicar o escalonamento de features para padronizar as colunas numéricas. Embora Random Forest e LightGBM sejam menos sensíveis à escala, é uma boa prática para muitos outros modelos e ajuda na generalização.

In [3]:
# Dividir os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de X_test: {X_test.shape}")
print(f"Formato de y_train: {y_train.shape}")
print(f"Formato de y_test: {y_test.shape}")

# Escalonar as features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados pré-processados e escalonados com sucesso!")

Formato de X_train: (455, 30)
Formato de X_test: (114, 30)
Formato de y_train: (455,)
Formato de y_test: (114,)
Dados pré-processados e escalonados com sucesso!


### 3. Treinamento e Ajuste de Hiperparâmetros: Random Forest

Utilizaremos `RandomizedSearchCV` para encontrar uma boa combinação de hiperparâmetros para o modelo Random Forest, avaliando o desempenho com a métrica AUC.

In [4]:
# Definir o classificador Random Forest
rf_classifier = RandomForestClassifier(random_state=42)

# Definir o espaço de busca para RandomizedSearchCV
param_distributions_rf = {
    'n_estimators': np.arange(50, 200, 25),  # Número de árvores
    'max_depth': [None, 10, 20, 30],         # Profundidade máxima da árvore
    'min_samples_split': np.arange(2, 11, 2), # Mínimo de amostras para dividir um nó
    'min_samples_leaf': np.arange(1, 6, 1),   # Mínimo de amostras por folha
    'max_features': ['sqrt', 'log2']         # Número de features a considerar em cada split
}

# Configurar RandomizedSearchCV
random_search_rf = RandomizedSearchCV(
    estimator=rf_classifier,
    param_distributions=param_distributions_rf,
    n_iter=10, # Número de iterações (tente 50 ou 100 para um ajuste mais robusto)
    scoring='roc_auc', # Métrica de avaliação (AUC)
    cv=5, # Cross-validation de 5 folds
    verbose=1, # Nível de detalhe da saída
    random_state=42,
    n_jobs=-1 # Usar todos os cores disponíveis
)

# Executar o ajuste de hiperparâmetros
print("Iniciando Randomized Search para Random Forest...")
random_search_rf.fit(X_train_scaled, y_train)

print("Random Forest - Melhores hiperparâmetros:", random_search_rf.best_params_)
print("Random Forest - Melhor AUC (treino com CV):", random_search_rf.best_score_)

Iniciando Randomized Search para Random Forest...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Random Forest - Melhores hiperparâmetros: {'n_estimators': np.int64(150), 'min_samples_split': np.int64(6), 'min_samples_leaf': np.int64(1), 'max_features': 'log2', 'max_depth': 10}
Random Forest - Melhor AUC (treino com CV): 0.9892156862745098


### 4. Treinamento e Ajuste de Hiperparâmetros: LightGBM

Faremos o mesmo processo para o modelo LightGBM.

In [5]:
# Definir o classificador LightGBM
lgbm_classifier = LGBMClassifier(random_state=42, verbose=-1) # verbose=-1 para suprimir mensagens de log

# Definir o espaço de busca para RandomizedSearchCV
param_distributions_lgbm = {
    'n_estimators': np.arange(50, 200, 25),
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'num_leaves': np.arange(20, 41, 5), # Número máximo de folhas em uma árvore
    'max_depth': [-1, 5, 10, 15],       # Profundidade máxima da árvore (-1 significa sem limite)
    'min_child_samples': np.arange(10, 31, 5), # Mínimo de dados em uma folha
    'subsample': [0.7, 0.8, 0.9, 1.0],  # Proporção de amostras para o crescimento da árvore
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0] # Proporção de features para o crescimento da árvore
}

# Configurar RandomizedSearchCV
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm_classifier,
    param_distributions=param_distributions_lgbm,
    n_iter=20, # Número de iterações
    scoring='roc_auc', # Métrica de avaliação (AUC)
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Executar o ajuste de hiperparâmetros
print("Iniciando Randomized Search para LightGBM...")
random_search_lgbm.fit(X_train_scaled, y_train)

print("LightGBM - Melhores hiperparâmetros:", random_search_lgbm.best_params_)
print("LightGBM - Melhor AUC (treino com CV):", random_search_lgbm.best_score_)

Iniciando Randomized Search para LightGBM...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
LightGBM - Melhores hiperparâmetros: {'subsample': 0.7, 'num_leaves': np.int64(30), 'n_estimators': np.int64(125), 'min_child_samples': np.int64(15), 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
LightGBM - Melhor AUC (treino com CV): 0.9936016511867904


### 5. Avaliação Final dos Modelos no Conjunto de Teste

Finalmente, vamos avaliar o desempenho dos melhores modelos (com os hiperparâmetros ajustados) no conjunto de teste, usando a métrica AUC.

In [6]:
# Melhor modelo Random Forest
best_rf_model = random_search_rf.best_estimator_
y_pred_proba_rf = best_rf_model.predict_proba(X_test_scaled)[:, 1]
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f"AUC do Random Forest no conjunto de teste: {auc_rf:.4f}")

# Melhor modelo LightGBM
best_lgbm_model = random_search_lgbm.best_estimator_
y_pred_proba_lgbm = best_lgbm_model.predict_proba(X_test_scaled)[:, 1]
auc_lgbm = roc_auc_score(y_test, y_pred_proba_lgbm)
print(f"AUC do LightGBM no conjunto de teste: {auc_lgbm:.4f}")

print("\nComparação:")
if auc_rf > auc_lgbm:
    print("Random Forest teve um desempenho ligeiramente melhor no conjunto de teste.")
elif auc_lgbm > auc_rf:
    print("LightGBM teve um desempenho ligeiramente melhor no conjunto de teste.")
else:
    print("Ambos os modelos tiveram desempenho similar no conjunto de teste.")

AUC do Random Forest no conjunto de teste: 0.9931
AUC do LightGBM no conjunto de teste: 0.9888

Comparação:
Random Forest teve um desempenho ligeiramente melhor no conjunto de teste.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
